# HW5 Q1: ResNet-20 quantization-aware training (QAT)
**(A) 4-bit:** train `resnet20_quant` (4-bit weights and activations in every BasicBlock conv and both 1×1 downsample convs; the first conv and the last FC layer stay full precision), then for `model.layer1[0].conv1` find `x_int` and `w_int` and recover the layer output from an integer convolution.

**For the submission:** after training, do *Kernel → Restart & Clear Output*, then run every cell **except** the training cell(s).

## Setup

In [ ]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "resnet20_quant"
model = resnet20_quant()   # 4-bit QuantConv2d in every BasicBlock and both 1x1 downsample convs

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

# (A) 4-bit quantization

## Training (run once, then Restart & Clear Output; skip this cell for the submission)

In [ ]:
# QAT training. Run this cell ONCE to train and save the best checkpoint to result/resnet20_quant/.
# Then do Kernel -> Restart & Clear Output, and for the submission run every cell EXCEPT this one.

lr = 0.1
weight_decay = 5e-4
epochs = 100
best_prec = 0

model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)

# The clipping values (wgt_alpha, act_alpha) are trained too, but get no weight decay: decay would
# keep shrinking the clipping range. weight_q is only a copy of the last quantized weights that the
# layer stores in forward(), never used in training, so it is left out of the optimizer.
alpha_params = [p for n, p in model.named_parameters() if 'alpha' in n]
other_params = [p for n, p in model.named_parameters() if 'alpha' not in n and 'weight_q' not in n]
optimizer = torch.optim.SGD([{'params': other_params, 'weight_decay': weight_decay},
                             {'params': alpha_params, 'weight_decay': 0}],
                            lr=lr, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)

for epoch in range(0, epochs):
    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()

## Final evaluation on the test set

In [ ]:
# Reload the saved best checkpoint and evaluate it on the test set.
fdir = 'result/'+str(model_name)+'/model_best.pth.tar'
checkpoint = torch.load(fdir, map_location=device)
model.load_state_dict(checkpoint['state_dict'])

criterion = nn.CrossEntropyLoss().to(device)
model.to(device)
model.eval()

prec = validate(testloader, model, criterion)
model.show_params()   # trained clipping values (alpha) of every QuantConv2d

## x_int and w_int for the 2nd conv layer: `model.layer1[0].conv1`
Capture the layer's input with a forward **pre**-hook and its output with a forward hook, on a batch of test images.

In [ ]:
layer = model.layer1[0].conv1
print(layer, '| bit width:', layer.bit)

hooked = {}
def save_input(module, input):
    hooked['x'] = input[0].detach().clone()     # clone: the ReLU before this layer is in-place
def save_output(module, input, output):
    hooked['out'] = output.detach().clone()

handles = [layer.register_forward_pre_hook(save_input),
           layer.register_forward_hook(save_output)]

images, labels = next(iter(testloader))
model.eval()
with torch.no_grad():
    model(images.to(device))   # fills the hooks; forward() also stores this layer's quantized weights in layer.weight_q
for h in handles:
    h.remove()

print('input x:', hooked['x'].shape, '  output:', hooked['out'].shape)

**w_int.** `weight_q` is stored in *normalized* units (the weights minus their mean, divided by their std, then quantized), and that is what the layer convolves with, so `w_int` comes from `weight_q`, not from the raw `layer.weight`.
Weights are signed, with 1 bit for the sign, so $\delta_w = \alpha_w / (2^{b-1}-1)$ and 4-bit gives integers in $-7..7$.

In [ ]:
w_bit = 4
weight_q = layer.weight_q.detach()                   # quantized weights stored during the forward pass
w_alpha = layer.weight_quant.wgt_alpha.detach()      # trained weight clipping value
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)           # signed: 7 levels per sign
weight_int = weight_q / w_delta

print('w_alpha =', round(w_alpha.item(), 4), '  w_delta =', round(w_delta.item(), 4))
print('max distance from an integer:', (weight_int - weight_int.round()).abs().max().item())
weight_int = weight_int.round()                      # remove float rounding noise (~1e-6)
print('w_int range:', int(weight_int.min()), 'to', int(weight_int.max()))
print(weight_int[0])                                 # first output filter: clean integers

**x_int.** The layer quantizes its input inside `forward()`, so the hooked input is still floating point. Quantize it with the layer's own activation quantizer and clipping value.
Activations come after a ReLU, so they are unsigned: $\delta_x = \alpha_x / (2^b - 1)$ and 4-bit gives integers in $0..15$.

In [ ]:
x_bit = 4
x = hooked['x']                                      # input of layer1[0].conv1 (floating point)
x_alpha = layer.act_alpha.detach()                   # trained activation clipping value
x_delta = x_alpha / (2 ** x_bit - 1)                 # unsigned: 15 levels

act_quant_fn = act_quantization(x_bit)               # same activation quantizer the layer uses
with torch.no_grad():
    x_q = act_quant_fn(x, x_alpha)
x_int = x_q / x_delta

print('x_alpha =', round(x_alpha.item(), 4), '  x_delta =', round(x_delta.item(), 4))
print('max distance from an integer:', (x_int - x_int.round()).abs().max().item())
x_int = x_int.round()
print('x_int range:', int(x_int.min()), 'to', int(x_int.max()))
print(x_int[0, 0, :6, :6])                           # corner of the first channel of the first image

## Recovered psum from an integer convolution
$$psum_{recovered} = conv(x_{int}, w_{int}) \times \delta_x \times \delta_w$$
compared with the layer's own output captured by the forward hook (should be ≈ 0, up to float rounding).

In [ ]:
conv_int = torch.nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_int.weight = torch.nn.Parameter(weight_int)

with torch.no_grad():
    psum_int = conv_int(x_int)                       # integer partial sums (exact in float32: |psum| <= 144*15*7)
    psum_recovered = psum_int * x_delta * w_delta    # back to the layer's scale

out = hooked['out']
diff = (psum_recovered - out).abs()                  # absolute, so + and - errors cannot cancel
print('psum_int is integer:', torch.equal(psum_int, psum_int.round()))
print('vs layer output: mean abs difference =', diff.mean().item(), '  max =', diff.max().item())
print('            mean abs layer output =', out.abs().mean().item())

For reference, the comparison in [HW_Code5]: the same quantized weights applied to the **floating-point** input `x`.
This gap is the activation quantization error, not a calculation error, so it depends on the trained model and has no required value.

In [ ]:
with torch.no_grad():
    out_float_x = F.conv2d(x, weight_q, stride=1, padding=1)
print('vs float-input reference: mean abs difference =', (psum_recovered - out_float_x).abs().mean().item())

# (B) 2-bit quantization
`QuantConv2d` fixes `self.bit = 4`, so the 2-bit model uses two new files:
- `models/quant_layer_2bit.py`: `QuantConv2d_nbit`, the same layer with the bit width as an argument (here 2), so both quantizers are built with 2 bits
- `models/resnet_quant_2bit.py`: `resnet20_quant_2bit`, with every BasicBlock conv and 1×1 downsample conv 2-bit, and **the first conv layer quantized too** (its `act_alpha` starts at 2.0 instead of 8.0, to match the normalized image range of about −2 to 2.1). The last FC layer stays full precision.

In [ ]:
from models.resnet_quant_2bit import resnet20_quant_2bit
from models.quant_layer_2bit import QuantConv2d_nbit

model_name_2bit = "resnet20_quant_2bit"
model_2bit = resnet20_quant_2bit().to(device)

print(model_2bit.conv1)               # first conv layer: now a 2-bit quantized conv
print(model_2bit.layer1[0].conv1)
print(model_2bit.fc)                  # last FC layer: still full precision
qconvs = [m for m in model_2bit.modules() if isinstance(m, QuantConv2d_nbit)]
print('quantized conv layers:', len(qconvs), ' bit widths:', sorted(set(m.bit for m in qconvs)))
print('first conv initial act_alpha:', model_2bit.conv1.act_alpha.item())

## 2-bit training (run once, then Restart & Clear Output; skip this cell for the submission)

In [ ]:
# 2-bit QAT training. Run this cell ONCE to train and save the best checkpoint to result/resnet20_quant_2bit/.
# Then do Kernel -> Restart & Clear Output, and for the submission run every cell EXCEPT the training cells.

# Start from the trained 4-bit model of part (A) (progressive quantization: FP -> 4-bit -> 2-bit).
# Its weights already work well under quantization, so 2-bit QAT converges faster than from scratch.
# The first conv layer's quantizer is new (not in the 4-bit model), so its alphas keep their initial values.
init_from_4bit = True

if init_from_4bit:
    lr, epochs = 0.05, 60
    ckpt_4bit = torch.load('result/'+str(model_name)+'/model_best.pth.tar', map_location=device)
    missing, unexpected = model_2bit.load_state_dict(ckpt_4bit['state_dict'], strict=False)
    print('started from the 4-bit model. New in the 2-bit model:', missing, ' unused:', unexpected)
else:
    lr, epochs = 0.1, 100

weight_decay = 5e-4
best_prec = 0

model_2bit = model_2bit.to(device)
criterion = nn.CrossEntropyLoss().to(device)

# same optimizer setup as (A): no weight decay on the clipping values, stored weight_q not optimized
alpha_params = [p for n, p in model_2bit.named_parameters() if 'alpha' in n]
other_params = [p for n, p in model_2bit.named_parameters() if 'alpha' not in n and 'weight_q' not in n]
optimizer = torch.optim.SGD([{'params': other_params, 'weight_decay': weight_decay},
                             {'params': alpha_params, 'weight_decay': 0}],
                            lr=lr, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

fdir = 'result/'+str(model_name_2bit)
if not os.path.exists(fdir):
    os.makedirs(fdir)

for epoch in range(0, epochs):
    train(trainloader, model_2bit, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model_2bit, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model_2bit.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()

## 2-bit final evaluation on the test set

In [ ]:
fdir = 'result/'+str(model_name_2bit)+'/model_best.pth.tar'
checkpoint = torch.load(fdir, map_location=device)
model_2bit.load_state_dict(checkpoint['state_dict'])

criterion = nn.CrossEntropyLoss().to(device)
model_2bit.to(device)
model_2bit.eval()

prec_2bit = validate(testloader, model_2bit, criterion)
print(model_2bit.conv1)       # the first conv layer is quantized
model_2bit.show_params()      # trained clipping values (alpha) of every quantized conv

## Check: the 2-bit model really uses 2-bit integers
Same steps as in (A), wrapped in a function: 2-bit weights should be integers in −1..1 and 2-bit activations in 0..3.
(The first conv layer's input is the normalized image, which can be negative, and the activation quantizer only clips from above, so its `x_int` can also be negative there.)

In [ ]:
def integer_check(model, layer, images, bit):
    hooked = {}
    h1 = layer.register_forward_pre_hook(lambda m, inp: hooked.update(x=inp[0].detach().clone()))
    h2 = layer.register_forward_hook(lambda m, inp, out: hooked.update(out=out.detach().clone()))
    model.eval()
    with torch.no_grad():
        model(images.to(device))
    h1.remove(); h2.remove()

    w_delta = layer.weight_quant.wgt_alpha.detach() / (2 ** (bit - 1) - 1)   # signed weights
    x_delta = layer.act_alpha.detach() / (2 ** bit - 1)                        # unsigned activations
    w_int = (layer.weight_q.detach() / w_delta).round()
    with torch.no_grad():
        x_int = (act_quantization(bit)(hooked['x'], layer.act_alpha.detach()) / x_delta).round()
        psum = F.conv2d(x_int, w_int, stride=layer.stride, padding=layer.padding) * x_delta * w_delta
    print(f'  w_int values: {sorted(w_int.unique().int().tolist())}   x_int values: {sorted(x_int.unique().int().tolist())}')
    print(f'  recovered psum vs layer output: mean abs difference = {(psum - hooked["out"]).abs().mean().item():.2e}')

print('first conv layer (model_2bit.conv1):')
integer_check(model_2bit, model_2bit.conv1, images, bit=2)
print('2nd conv layer (model_2bit.layer1[0].conv1):')
integer_check(model_2bit, model_2bit.layer1[0].conv1, images, bit=2)